# Multiple DAC units — offline routing and safety tests

All fake-unit behavior and assertions are written in these cells. No instrument is contacted. Tests cover stable global numbering, duplicate-address/USB rejection, per-unit errors, parallel operation, recovery preemption, GUI routing, stale/offline rejection and request deduplication. They do **not** claim that a second physical unit has been tested.

In [ ]:
import sys,time,threading,socket
from pathlib import Path
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'malab/instruments/dac_array.py').exists())
sys.path.insert(0,str(root))
from malab.instruments.voltsource import DACArray
from malab.instruments.dac_array import UnitOperationError
from malab.instruments.ad5780 import ChannelCalibration
from malab.instruments._line_socket import InstrumentConnectionError
from malab.dashboard.controller import Dashboard
@contextmanager
def raises(kind):
    try:yield
    except kind as e:print(type(e).__name__+': '+str(e))
    else:raise AssertionError('Expected '+kind.__name__)
class Fake:
    def __init__(self,**kw):
        self.address=kw['address'];self.bench_port=kw['bench_port'];self.connected=False;self.enabled=False
        self.calls=[];self.codes=[131072]*4;self.bad=False;self.waiting=threading.Event();self.release=threading.Event()
        self.state='IDLE'
    def connect(self):
        if self.bad:raise InstrumentConnectionError('offline')
        self.connected=True;self.calls.append(('connect',))
    def disconnect(self):self.connected=False;self.calls.append(('disconnect',))
    def close(self):self.disconnect()
    def get_id(self):return 'AD5780'
    def initialize(self,ch=None):self.calls.append(('initialize',ch))
    def get_calibration(self,ch,**kw):return ChannelCalibration(-10000000000,76293945,131072,False)
    def voltage_to_code(self,ch,v):return self.get_calibration(ch).code(v)
    def code_to_voltage(self,ch,c):return self.get_calibration(ch).voltage(c)
    def set_voltage(self,ch,v):
        if self.bad:raise InstrumentConnectionError('offline; outcome unknown')
        self.calls.append(('set',ch,v));self.codes[ch-1]=self.voltage_to_code(ch,v)
        return self.code_to_voltage(ch,self.codes[ch-1])
    def get_code(self,ch):return self.codes[ch-1]
    def set_code(self,ch,c):self.codes[ch-1]=c;self.calls.append(('code',ch,c));return c
    def get_voltage(self,ch):return self.code_to_voltage(ch,self.get_code(ch))
    def get_status(self):return SimpleNamespace(state=self.state,channel=0 if self.state=='IDLE' else 4,target_code=131072)
    def get_output_codes(self,**kw):return [dict(channel=i+1,code=c,source='readback') for i,c in enumerate(self.codes)]
    def ramp(self,ch,v,speed,**kw):
        self.calls.append(('ramp',ch,v,speed));self.state='RUNNING'
        if kw.get('wait'):
            self.waiting.set();assert self.release.wait(2),'Test ramp release missing'
        return SimpleNamespace(state='ACCEPTED',channel=ch,target_code=self.voltage_to_code(ch,v))
    def safe_zero(self,**kw):
        if not self.bench_port:raise ValueError('No USB recovery port')
        self.calls.append(('safe_zero',));self.release.set();return 'accepted'
    def network_reset(self,**kw):self.calls.append(('network',));return 'accepted'
    def get_recovery_status(self):return 'READY'
    def abort_ramp(self):self.disconnect();self.release.set()
addresses=['192.0.2.10','192.0.2.20','192.0.2.30'] # Documentation IPs; fake factory only.


## M01 — Immutable ordering, exact local mapping, no commands on construction

In [ ]:
a=DACArray(addresses,bench_ports=['COM5','COM6',None],dac_factory=Fake)
assert all(d.calls==[] for d in a.devices)
assert a.channel_count==12
for ch in range(1,13):
    loc=a.locate(ch)
    assert (loc.unit,loc.local_channel,loc.address)==((ch-1)//4+1,(ch-1)%4+1,addresses[(ch-1)//4])
addresses.reverse()
assert a.locate(1).address=='192.0.2.10','Caller list mutation changed mapping'
a.set_voltage(5,.05);a.set_code(12,131000)
assert a.devices[1].calls==[('set',1,.05)]
assert a.devices[2].calls==[('code',4,131000)] and a.devices[0].calls==[]
r=a.ramp(8,.1,.01,wait=False)
assert r['channel']==8 and r['local_channel']==4 and r['unit']==2
assert a.get_output_codes(unit=2)[0]['channel']==5
assert a.get_status(unit=2)['channel']==8


In [ ]:
for bad in [[], '192.0.2.1', ['192.0.2.1','192.0.2.1:23'], ['not-an-ip'], ['192.0.2.1:0']]:
    with raises(ValueError):DACArray(bad,dac_factory=Fake)
with raises(ValueError):DACArray(['192.0.2.1','192.0.2.2'],bench_ports=['COM5','com5'],dac_factory=Fake)
for ch in [0,13,-1,True,1.2,'1']:
    with raises(ValueError):a.set_voltage(ch,0)


## M03 — A failed unit stays in its slot; outputs are not retried or rerouted

In [ ]:
a.devices[1].bad=True
results=a.connect()
assert results[1].ok and not results[2].ok and results[3].ok
assert a.locate(9).unit==3 and a.locate(5).unit==2
before=[list(d.calls) for d in a.devices]
with raises(UnitOperationError):a.set_voltage(5,.2)
assert [d.calls for d in a.devices]==before
a.set_voltage(9,.2);assert a.devices[2].calls[-1]==('set',1,.2)
a.devices[1].bad=False
a.network_reset(unit=2);assert a.devices[1].calls[-1]==('network',)
assert a.devices[0].calls==before[0]
results=a.safe_zero_all()
assert results[1].ok and results[2].ok and not results[3].ok
assert 'No USB' in results[3].error


## M04 — A blocking operation on unit 1 leaves unit 2 and USB recovery usable

In [ ]:
a=DACArray(['192.0.2.1','192.0.2.2'],bench_ports=['COM5','COM6'],dac_factory=Fake)
with ThreadPoolExecutor(max_workers=2) as pool:
    first=pool.submit(a.ramp,1,.1,.01,wait=True)
    try:
        assert a.devices[0].waiting.wait(.5)
        second=pool.submit(a.set_voltage,5,.05)
        assert abs(second.result(timeout=.5)-.05)<.0001
        a.safe_zero(unit=1) # Must not wait for the normal operation lock.
        first.result(timeout=.5)
    finally:a.devices[0].release.set()


## M05 — GUI routing and request IDs cannot cross units

In [ ]:
d=Dashboard(['192.0.2.1','192.0.2.2'],bench_ports=['COM5','COM6'],meter_channel=8,dac_factory=Fake,meter_address='192.0.2.71')
for u in d.units:u.poll_dac()
s=d.snapshot();assert [c['channel'] for c in s['channels']]==list(range(1,9))
assert s['meter']['channel']==8 and s['channels'][7]['local_channel']==4
request=dict(action='set',channel=5,unit=2,voltage=.05,request_id='once')
d.submit(request);d.submit(request)
assert d.units[0].commands.empty() and d.units[1].commands.qsize()==1
local=d.units[1].commands.get();assert local['channel']==1
d.units[1].action(local);d.units[1].complete(local)
d.submit(request);assert d.units[1].commands.empty()
assert d.units[1].dac.calls[-2:]==[('initialize',1),('set',1,.05)]
with raises(ValueError):d.submit(dict(request,channel=1,unit=1))
with raises(ValueError):d.submit(dict(action='set',channel=5,unit=1,voltage=0,request_id='wrong-unit'))
with raises(ValueError):d.submit(dict(action='safe_zero',request_id='missing-unit'))
d.submit(dict(action='safe_zero',unit=2,request_id='unit-two-only'))
assert d.units[0].commands.empty()
assert d.units[1].commands.get()['action']=='safe_zero'
d.units[1].complete(dict(action='safe_zero',request_id='unit-two-only'))


## M06 — Offline, stale, busy or latched unit 1 does not disable unit 2

In [ ]:
for mode in ('failed','stale','busy','latched'):
    d=Dashboard(['192.0.2.1','192.0.2.2'],bench_ports=[None,None],meter_channel=4,dac_factory=Fake,meter_address='192.0.2.71')
    for u in d.units:u.poll_dac()
    if mode=='failed':d.units[0].state['network']['state']='failed'
    elif mode=='stale':d.units[0].state['network']['last_ok']=time.time()-10
    elif mode=='busy':d.units[0].state['ramp']['state']='RUNNING'
    elif mode=='latched':d.units[0].state['recovery']['state']='LATCHED'
    with raises(ValueError):d.submit(dict(action='set',channel=1,voltage=0,request_id='blocked'))
    d.submit(dict(action='set',channel=5,voltage=0,request_id='other-unit'))
    assert d.units[0].commands.empty() and d.units[1].commands.qsize()==1
    assert len(d.snapshot()['channels'])==8
with raises(ValueError):d.submit(dict(action='safe_zero',unit=2,request_id='no-usb'))
assert all(('safe_zero',) not in u.dac.calls for u in d.units)


## M07 — Independent live worker threads with one simulated slow/offline unit

In [ ]:
d=Dashboard(['192.0.2.1','192.0.2.2'],bench_ports=[None,None],dac_factory=Fake,meter_address='192.0.2.71')
entered=threading.Event();release=threading.Event()
def blocked_connect():
    entered.set();release.wait(2)
    raise InstrumentConnectionError('simulated outage')
d.units[0].dac.connect=blocked_connect
for u in d.units:u.start()
try:
    assert entered.wait(.5)
    deadline=time.monotonic()+1
    while time.monotonic()<deadline and d.units[1].snapshot()['network']['state']!='good':time.sleep(.01)
    assert d.units[1].snapshot()['network']['state']=='good'
    assert d.units[0].snapshot()['network']['last_ok'] is None
    d.submit(dict(action='set',channel=8,voltage=.05,request_id='while-first-offline'))
    deadline=time.monotonic()+1
    while time.monotonic()<deadline and d.units[1].snapshot()['pending']:time.sleep(.01)
    assert d.units[1].dac.calls[-1]==('set',4,.05)
finally:
    release.set();d.close()
print('All 7 multi-unit routing/safety groups passed; no physical second unit tested.')
